<head>
    <style>
        .md-typeset h2 {
            margin:0;
            }
        .md-typeset h3 {
            margin:0;
            }
        .jupyter-wrapper table.dataframe tr, .jupyter-wrapper table.dataframe th, .jupyter-wrapper table.dataframe td {
            text-align:left;
            }
        .jupyter-wrapper table.dataframe {
            table-layout: auto;
            }
    </style>        
</head>

# Value Discoveries 

## Introduction

Before reading this post, please watch the following talk by Aswath Damodaran.

<iframe width="560" height="315" src="https://www.youtube.com/embed/Z5chrxMuBoo" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

Borrowing the concept from Aswath Damodaran, we wil categorize firms as

* old firms: focus on ratio analysis as financial reports could tell us a lot about
this kind of firms
* mature firms: focus on ratio analysis and narrative 
* young firms: focus on narrative as valuing young firms is more like evaluating
their growth potential (there is not much to analyze from their financial reports)

Since we are using `python` to analyze their financial information, one should
be aware that only old and mature firms could be examined with this kind of tool.
For young firms, you need to do more research out of financial reports. 

In this post, we will focus on telling a story rather than presenting the code.
Therefore, many process will be done via wrapped `functions` and definitions of
those functions will be given at the end of the post.

In [11]:
# import packages
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import json
import random
sns.set_theme()
plt.style.use('seaborn-notebook')  # set the theme

# requests headers
headers = """
authority: data.sec.gov
method: GET
scheme: https
accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9
accept-encoding: gzip, deflate, br
accept-language: en-US,en;q=0.9
cache-control: max-age=0
sec-fetch-dest: document
sec-fetch-mode: navigate
sec-fetch-site: none
sec-fetch-user: ?1
upgrade-insecure-requests: 1
user-agent: Mozilla/5.0 (iPhone; CPU iPhone OS 13_2_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/13.0.3 Mobile/15E148 Safari/604.1 Edg/103.0.5060.114
"""

headers = headers.strip().split('\n')
# save it as dict
HEADERS = {x.split(':')[0].strip():
           ("".join(x.split(':')[1:])).strip().replace('//',
                                                       "://") for x in headers}

## EDGAR API Again 

In the last post, we ger ourselves familiar with `EDGAR API` from SEC. However,
we have not figured out how to tell a story of valuation from those account
items. In this post, we will try to value a firm by using its financial reports.

In our dataset we have the following sectors:

```python
sectors = ['Industrials', 'Finance', 'Consumer Discretionary', 'Health Care',
              'Real Estate', nan, 'Technology', 'Energy', 'Consumer Staples',
              'Miscellaneous', 'Utilities', 'Telecommunications',
              'Basic Materials']
```

In [8]:
# it firms dataset
it_sectors = ['Technology', 'Telecommunications']
it_firms = load_dataset(it_sectors)  
it_firms.head()

,Symbol,Name,Last Sale,Net Change,% Change,Market Cap,Country,IPO Year,Volume,Sector,Industry,CIK
17,AAOI,Applied Optoelectronics Inc. Common Stock,$1.57,-0.0400,-2.484%,4.338551e+07,United States,2013.0,2039,Technology,Semiconductors,0001158114
20,AAPL,Apple Inc. Common Stock,$145.4894,-0.0006,0.00%,2.522399e+12,United States,1980.0,23406341,Technology,Computer Manufacturing,0000320193
68,ACCD,Accolade Inc. Common Stock,$9.38,-0.1900,-1.985%,6.679445e+08,United States,2020.0,139680,Technology,Interactive Media,0001481646
74,ACEV,ACE Convergence Acquisition Corp. Class A Ordi...,$10.17,0.0000,0.00%,1.418947e+08,United States,2020.0,462,Technology,Semiconductors,0001813658
75,ACEVU,ACE Convergence Acquisition Corp. Unit,$10.15,-0.0001,-0.001%,0.000000e+00,United States,2020.0,400,Technology,Semiconductors,0001813658


In [10]:
# old, young, mature
it_old = filter_age(it_firms, 'old')
it_old.head()

,Symbol,Name,Last Sale,Net Change,% Change,Market Cap,Country,IPO Year,Volume,Sector,Industry,CIK
20,AAPL,Apple Inc. Common Stock,$145.4894,-0.0006,0.00%,2.522399e+12,United States,1980.0,23406341,Technology,Computer Manufacturing,0000320193
91,ACLS,Axcelis Technologies Inc. Common Stock,$52.9668,-0.8532,-1.585%,1.748986e+09,United States,2000.0,69120,Technology,Industrial Machinery/Components,0001113232
125,ADBE,Adobe Inc. Common Stock,$366.39,-5.5500,-1.492%,1.714705e+11,United States,1986.0,610085,Technology,Computer Software: Prepackaged Software,0000796343
156,ADTN,ADTRAN Holdings Inc. Common Stock,$19.30,-0.2800,-1.43%,9.480205e+08,United States,1994.0,117741,Telecommunications,Telecommunications Equipment,0000926282
373,ALLT,Allot Ltd. Ordinary Shares,$4.87,-0.0100,-0.205%,1.781809e+08,Israel,2006.0,16460,Telecommunications,Telecommunications Equipment,0001365767


## Python Scripts 

In [9]:
# load the dataset
def _sector_filter(sector_str, sector_conditions):
    if sector_str in sector_conditions:
        return True
    else:
        return False
    
def _match_cik(symbol, sec_tickers):
    for x in sec_tickers:
        if sec_tickers[x]['ticker'] == symbol:
            cik = str(sec_tickers[x]['cik_str'])
            cik_pre = '0'*(10-len(cik))
            return cik_pre+cik
        
def load_dataset(sector_list):
    # please download data from 
    # https://www.nasdaq.com/market-activity/stocks/screener
    stock_screener = pd.read_csv('./data/nasdaq_screener.csv')
    # filter out it firms
    it_industries = sector_list
    it_mask = stock_screener['Sector'].apply(lambda x: 
                                    _sector_filter(x, it_industries))
    it_firms = stock_screener[it_mask]
    # match cik numbers
    # read company tickers from sec
    r = requests.get("https://www.sec.gov/files/company_tickers.json")
    sec_tickers = r.json()
    temp = it_firms['Symbol'].apply(lambda x: _match_cik(x, sec_tickers))
    it_firms = it_firms.assign(CIK = temp)
    
    return it_firms

def _filter_age(ipo_year, firm_type):
    if firm_type == 'young':
        if ipo_year >= 2017:
            return True
        else:
            return False
    elif firm_type == 'mature':
        if ipo_year >= 2011 and ipo_year < 2017:
            return True
        else:
            return False
    else:
        if ipo_year <= 2010:
            return True
        else:
            return False
        
def filter_age(df, firm_type):
    mask = df['IPO Year'].apply(lambda x: _filter_age(x, firm_type))
    return df[mask]